# CellRegPy – Validate Alignment Demo

End-to-end cross-session cell registration pipeline, reproducing the
MATLAB `batchRunCellReg(folder_path, [], [], true)` validation workflow.

## What this notebook does
1. **Stage 1** — Load spatial footprints from suite2p sessions  
2. **Stage 2** — Align sessions using mean-image registration  
3. **Stage 3** — Fit probabilistic models (centroid distance + spatial correlation)  
4. **Stage 4** — Initial registration  
5. **Stage 5** — Final clustering and accuracy estimation  

Each stage produces MATLAB-matching diagnostic figures.

> **Note:** If a `CellReg.mat` file is not found in a session folder,
> footprints will be **auto-generated** from suite2p's `stat.npy`.

## 0 · Setup

In [ ]:
%matplotlib inline

import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# CellRegPy core
from cellregpy import (
    CellRegConfig,
    list_session_folders,
    get_cellreg_files,
    get_spatial_footprints,
    get_mean_image,
    normalize_footprints,
    adjust_fov_size,
    compute_footprint_projections,
    compute_centroids,
    compute_centroid_projections,
    estimate_num_bins,
    compute_data_distribution,
    compute_centroid_distances_model_custom,
    compute_spatial_correlations_model,
    compute_p_same,
    choose_best_model,
    initial_registration_centroid_distances_custom,
    initial_registration_spatial_corr,
    cluster_cells_matlab,
    estimate_registration_accuracy,
    MeanImageAligner,
)

# CellRegPy plotting
from cellregpy.plotting import (
    plot_session_projections,
    validate_alignment_deck,
    plot_x_y_displacements,
    plot_models,
    plot_cell_scores,
    plot_all_registered_projections,
    plot_init_registration,
    plot_pairwise_session_overlap,
    _extract_p_from_model_string,
    _compute_histogram_distribution,
)

print('CellRegPy loaded ✓')

## 0.1 · Configuration

Edit the cell below to point to your mouse data folder and set parameters.

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────
folder_path = r"D:\data\mouse_name"   # <── CHANGE THIS

microns_per_pixel     = 2.0   # suite2p default for 1x zoom
maximal_distance      = 14    # max centroid distance (µm)
p_same_threshold      = 0.5   # P(same cell) threshold
registration_approach = 'simple'
alignable_threshold   = 0.15  # min cross-correlation to declare alignable

SAVE_FIGURES = True           # save PNG figures to disk
# ─────────────────────────────────────────────────────────────────────

---
## Stage 1 — Load Spatial Footprints

In [ ]:
normalized_maximal_distance = maximal_distance / microns_per_pixel

# Find session folders
plane0_folders = list_session_folders(Path(folder_path))
print(f'Found {len(plane0_folders)} session folders')

# Validation mode: first 4 sessions
plane0_folders = plane0_folders[:4]
n_sessions = len(plane0_folders)
print(f'Using first {n_sessions} sessions')

# Find (or auto-generate) CellReg.mat files
sess_fovs = get_cellreg_files(plane0_folders, auto_generate=True)
print(f'Found {len(sess_fovs)} CellReg.mat files')
assert len(sess_fovs) >= 2, 'Need at least 2 sessions'

In [ ]:
# Setup output directories
results_root = Path(folder_path) / '1_CellReg'
fov_dir = results_root / 'FOV1'
figures_directory = fov_dir / 'Figures'
results_directory = fov_dir / 'Results'
if SAVE_FIGURES:
    for d in [results_root, fov_dir, figures_directory, results_directory]:
        d.mkdir(parents=True, exist_ok=True)
fig_dir = str(figures_directory) if SAVE_FIGURES else None

In [ ]:
# Load spatial footprints
spatial_footprints_raw = []
for sf in sess_fovs:
    fp = get_spatial_footprints(sf)
    spatial_footprints_raw.append(fp)
    print(f'  {Path(sf).parent.parent.parent.name}: '
          f'{fp.shape[0]} cells, {fp.shape[1]}×{fp.shape[2]} FOV')

# Compute projections
footprints_projections = compute_footprint_projections(spatial_footprints_raw)

# Plot Stage 1
plot_session_projections(footprints_projections, fig_dir)

---
## Stage 2 — Align Sessions via Mean Images

In [ ]:
# Load mean images
mean_images = []
for pf in plane0_folders:
    mi = get_mean_image(pf, apply_drift_correction=True)
    mean_images.append(mi)
    print(f'  {Path(pf).parent.parent.name}: {mi.shape}')

# Normalize + Adjust FOV
normalized_fps = normalize_footprints(spatial_footprints_raw)
adjusted_fps, adjusted_fov, adj_x, adj_y, padding = adjust_fov_size(normalized_fps)
del normalized_fps

adjusted_projections = compute_footprint_projections(adjusted_fps)
centroid_locations = compute_centroids(adjusted_fps, microns_per_pixel)
centroid_projections = compute_centroid_projections(centroid_locations, adjusted_fps)

In [ ]:
from skimage import transform as sktransform

reference_session_index = 0
aligner = MeanImageAligner()

aligned_fps = [None] * n_sessions
aligned_centroid_locations = [None] * n_sessions
alignment_translations = np.zeros((3, n_sessions))
maximal_cross_correlation = np.zeros(n_sessions)

for i in range(n_sessions):
    if i == reference_session_index:
        aligned_fps[i] = adjusted_fps[i]
        aligned_centroid_locations[i] = centroid_locations[i]
        maximal_cross_correlation[i] = 1.0
    else:
        _, method, peak, tform, _, _ = aligner.align(
            mean_images[reference_session_index],
            mean_images[i],
            filter_mode='highpass',
            outlier_mode='off'
        )
        maximal_cross_correlation[i] = peak

        if tform is not None and peak >= alignable_threshold:
            params = tform.params
            alignment_translations[0, i] = params[0, 2]
            alignment_translations[1, i] = params[1, 2]
            angle_rad = np.arctan2(params[1, 0], params[0, 0])
            alignment_translations[2, i] = np.degrees(angle_rad)

            n_cells = adjusted_fps[i].shape[0]
            aligned = np.zeros_like(adjusted_fps[i])
            for c in range(n_cells):
                aligned[c] = sktransform.warp(
                    adjusted_fps[i][c], tform.inverse,
                    output_shape=adjusted_fps[i][c].shape,
                    order=1, preserve_range=True, mode='constant', cval=0.0)
            aligned_fps[i] = aligned

            cents = centroid_locations[i]
            if len(cents) > 0:
                coords = np.column_stack([cents[:, 0], cents[:, 1], np.ones(len(cents))])
                transformed = (tform.params @ coords.T).T
                aligned_centroid_locations[i] = transformed[:, :2]
            else:
                aligned_centroid_locations[i] = cents

            print(f'  Session {i+1}: peak={peak:.3f}, '
                  f'dx={alignment_translations[0,i]:.1f}, '
                  f'dy={alignment_translations[1,i]:.1f}, '
                  f'rot={alignment_translations[2,i]:.2f}°')
        else:
            aligned_fps[i] = adjusted_fps[i]
            aligned_centroid_locations[i] = centroid_locations[i]
            print(f'  Session {i+1}: peak={peak:.3f} (below threshold)')

corrected_projections = compute_footprint_projections(aligned_fps)

In [ ]:
# Alignment validation figures
session_names = [str(pf) for pf in plane0_folders]
validate_alignment_deck(
    mean_images,
    adjusted_projections,
    corrected_projections,
    reference_session_index=reference_session_index,
    alignment_translations=alignment_translations,
    scores=maximal_cross_correlation,
    out_dir=fig_dir,
    session_names=session_names,
)

---
## Stage 3 — Probabilistic Models

In [ ]:
# 3a — Data distribution
number_of_bins, _ = estimate_num_bins(adjusted_fps, normalized_maximal_distance)
centers_of_bins = (
    np.linspace(0, normalized_maximal_distance, number_of_bins, dtype=np.float64),
    np.linspace(0, 1, number_of_bins, dtype=np.float64),
)

data_dist = compute_data_distribution(
    aligned_fps, aligned_centroid_locations, normalized_maximal_distance,
)

# Plot x,y displacements
plot_x_y_displacements(
    data_dist['neighbors_x_displacements'],
    data_dist['neighbors_y_displacements'],
    microns_per_pixel, normalized_maximal_distance,
    number_of_bins, centers_of_bins, fig_dir,
)
print('Stage 3a ✓')

In [ ]:
# 3b — Fit models
centroid_centers = centers_of_bins[0]
corr_centers = centers_of_bins[1]

(p_same_given_centroid_distance,
 centroid_same_model, centroid_diff_model, centroid_mixture_model,
 centroid_intersection, centroid_best_str, centroid_mse
) = compute_centroid_distances_model_custom(
    data_dist['neighbors_centroid_distances'],
    number_of_bins, centers_of_bins,
    microns_per_pixel=microns_per_pixel,
)
p_centroid = _extract_p_from_model_string(centroid_best_str)

(p_same_given_spatial_correlation,
 spatial_same_model, spatial_diff_model, spatial_mixture_model,
 spatial_intersection, spatial_best_str, spatial_mse
) = compute_spatial_correlations_model(
    data_dist['neighbors_spatial_correlations'],
    number_of_bins, centers_of_bins,
)
p_spatial = _extract_p_from_model_string(spatial_best_str)

# Histogram distributions
centroid_distribution = _compute_histogram_distribution(
    data_dist['neighbors_centroid_distances'],
    centroid_centers, number_of_bins, scale=microns_per_pixel)
spatial_distribution = _compute_histogram_distribution(
    data_dist['neighbors_spatial_correlations'][
        data_dist['neighbors_spatial_correlations'] >= 0
    ], corr_centers, number_of_bins, scale=1.0)

# Plot models
plot_models(
    centroid_distances_model_parameters=np.array([p_centroid]),
    NN_centroid_distances=data_dist['NN_centroid_distances'],
    NNN_centroid_distances=data_dist['NNN_centroid_distances'],
    centroid_distances_distribution=centroid_distribution,
    centroid_distances_model_same_cells=centroid_same_model,
    centroid_distances_model_different_cells=centroid_diff_model,
    centroid_distances_model_weighted_sum=centroid_mixture_model,
    centroid_distance_intersection=centroid_intersection,
    centers_of_bins_dist=centroid_centers,
    spatial_correlations_model_parameters=np.array([p_spatial]),
    NN_spatial_correlations=data_dist['NN_spatial_correlations'],
    NNN_spatial_correlations=data_dist['NNN_spatial_correlations'],
    spatial_correlations_distribution=spatial_distribution,
    spatial_correlations_model_same_cells=spatial_same_model,
    spatial_correlations_model_different_cells=spatial_diff_model,
    spatial_correlations_model_weighted_sum=spatial_mixture_model,
    spatial_correlation_intersection=spatial_intersection,
    centers_of_bins_corr=corr_centers,
    microns_per_pixel=microns_per_pixel,
    maximal_distance=normalized_maximal_distance,
    out_dir=fig_dir,
)
print('Stage 3b ✓')

In [ ]:
best_model_string = choose_best_model(
    centroid_mse, spatial_mse,
    centroid_intersection=centroid_intersection,
    corr_intersection=spatial_intersection,
)
print(f'Best model: {best_model_string}')

---
## Stage 4 — Initial Registration

In [ ]:
p_same_centroid, p_same_spatial = compute_p_same(
    data_dist['all_to_all_centroid_distances'],
    data_dist['all_to_all_spatial_correlations'],
    centers_of_bins,
    p_same_given_centroid_distance,
    p_same_given_spatial_correlation,
)

initial_registration_type = best_model_string

if initial_registration_type == 'Spatial correlation':
    initial_threshold = spatial_intersection if np.isfinite(spatial_intersection) else 0.65
    (cell_to_index_map, registered_cells_metric,
     non_registered_cells_metric, _) = initial_registration_spatial_corr(
        aligned_fps, aligned_centroid_locations,
        maximal_distance=normalized_maximal_distance,
        spatial_correlation_threshold=initial_threshold)
else:
    initial_threshold = centroid_intersection if np.isfinite(centroid_intersection) else 5.0
    normalized_threshold = initial_threshold / microns_per_pixel
    (cell_to_index_map, registered_cells_metric,
     non_registered_cells_metric, _) = initial_registration_centroid_distances_custom(
        aligned_centroid_locations,
        maximal_distance=normalized_maximal_distance,
        centroid_distance_threshold=normalized_threshold)

print(f'{cell_to_index_map.shape[0]} clusters, threshold={initial_threshold:.3f}')

plot_init_registration(
    cell_to_index_map, number_of_bins, aligned_fps,
    initial_registration_type,
    registered_cells_metric, non_registered_cells_metric,
    microns_per_pixel=microns_per_pixel,
    maximal_distance=normalized_maximal_distance,
    out_dir=fig_dir,
)
print('Stage 4 ✓')

---
## Stage 5 — Final Registration

In [ ]:
if best_model_string == 'Spatial correlation':
    all_to_all_p_same = p_same_spatial
else:
    all_to_all_p_same = p_same_centroid

(optimal_cell_to_index_map,
 registered_cells_centroids,
 cluster_scores_dict) = cluster_cells_matlab(
    cell_to_index_map=cell_to_index_map,
    all_to_all_p_same=all_to_all_p_same,
    all_to_all_indexes=data_dist['all_to_all_indexes'],
    maximal_distance=normalized_maximal_distance,
    registration_threshold=p_same_threshold,
    centroid_locations=aligned_centroid_locations,
    registration_approach=registration_approach,
    transform_data=False,
)

p_same_vec, p_diff_vec, accuracy_scores = estimate_registration_accuracy(
    optimal_cell_to_index_map,
    all_to_all_p_same,
    data_dist['all_to_all_indexes'],
    threshold=p_same_threshold,
)

n_clusters_final = optimal_cell_to_index_map.shape[0]
if cluster_scores_dict:
    cell_scores = cluster_scores_dict.get('cell_scores', np.zeros(n_clusters_final))
    cell_scores_positive = cluster_scores_dict.get('cell_scores_positive', np.zeros(n_clusters_final))
    cell_scores_negative = cluster_scores_dict.get('cell_scores_negative', np.zeros(n_clusters_final))
    cell_scores_exclusive = cluster_scores_dict.get('cell_scores_exclusive', np.zeros(n_clusters_final))
    p_same_registered_pairs = cluster_scores_dict.get('p_same_registered_pairs', [])
else:
    cell_scores = np.zeros(n_clusters_final)
    cell_scores_positive = np.zeros(n_clusters_final)
    cell_scores_negative = np.zeros(n_clusters_final)
    cell_scores_exclusive = np.zeros(n_clusters_final)
    p_same_registered_pairs = []

print(f'{n_clusters_final} cell clusters in final registration')

In [ ]:
# Score distributions
plot_cell_scores(
    cell_scores, cell_scores_exclusive,
    cell_scores_positive, cell_scores_negative,
    p_same_registered_pairs, fig_dir,
)

# Final projections
plot_all_registered_projections(
    aligned_fps, optimal_cell_to_index_map,
    fig_dir, stage_label='Stage 5',
)

# Pairwise overlap
plot_pairwise_session_overlap(
    aligned_fps, optimal_cell_to_index_map, fig_dir,
)

print('Stage 5 ✓')

---
## Save Results

In [ ]:
np.save(results_directory / 'cell_to_index_map.npy', optimal_cell_to_index_map)
print(f'Saved: {results_directory / "cell_to_index_map.npy"}')

print(f'\n{"="*60}')
print(f'  Pipeline complete!')
print(f'  Total cells registered: {optimal_cell_to_index_map.shape[0]}')
print(f'  Best model: {best_model_string}')
print(f'{"="*60}')